## Claude Sonnet 4 con extended thinking

In [85]:
import pandas as pd
import datetime
import time


from google.colab import data_table
data_table.enable_dataframe_formatter()

from google.colab import userdata
userdata.get('API_KEY')
api_key=userdata.get('API_KEY')
serpapi_key=userdata.get('SERPAPI_KEY')
#API_KEY="INSERT YOUR API KEY"
THINKING_MODEL="claude-3-7-sonnet-20250219"
CLAUDE4="claude-sonnet-4-20250514"


In [ ]:
!pip install anthropic

In [55]:
from anthropic import Anthropic
client = Anthropic(
    api_key=api_key # API_KEY
)

In [18]:
def calcola_durata(time_fine, time_inizio):
  '''Funzione per calcolare la durata dell'inferenza'''
  durata = time_fine - time_inizio
  #print(f"Tempo impiegato: {durata:.2f} secondi")
  return durata

In [19]:
#dataframe to collect time length
data_for_df = []

#ALCUNI PROMPT DI ESEMPIO
dimostrazione_per_induzione="Dimostra per induzione che la somma dei primi n numeri naturali è n*(n+1)/2. Visualizza output in puro testo."
dimostrazione_potenze_due="Dimostra che per ogni numero naturale n ≥ 0, vale la seguente formula: sum(2^i, i=0..n) = 2^(n+1) - 1. Visualizza output in puro testo."
numeri_primi="Are there an infinite number of prime numbers such that n mod 4 == 3?"
analisi=f"Sei il responsabile dell'innovazione in un'azienda di medie dimensioni nel settore manifatturiero che è stata storicamente resistente al cambiamento. L'azienda è redditizia ma sta perdendo gradualmente quote di mercato rispetto ai concorrenti più innovativi. Il tuo CEO è pragmatico e attento ai costi, e considera l'innovazione come un rischio piuttosto che un'opportunità. \
Sviluppa una strategia completa per convincere il tuo responsabile ad investire in innovazione, considerando:\
1. Come presentare un business case convincente che dimostri il ROI dell'innovazione \
2. Quali dati e metriche utilizzare per supportare la tua argomentazione \
3. Una proposta di implementazione graduale che minimizzi i rischi \
4. Come affrontare le potenziali obiezioni basate su fallimenti passati di iniziative di cambiamento \
5. Un piano di comunicazione per ottenere il supporto dei dipendenti e altri stakeholder. \
Includi esempi concreti di aziende simili che hanno implementato con successo programmi di innovazione e i risultati che hanno ottenuto."
r_in_strawberry="quante r ci sono nella parola 'strawberry'?"
logica="Alice ha un fratello e 2 sorelle, quante sorelle ha il fratello di Alice?"


## CLAUDE 3.7 EXTENDED THINKING

In [ ]:
# chiamata a claude 3.7

start_time = time.time()
dt_object = datetime.datetime.fromtimestamp(start_time)
format1 = dt_object.strftime("%d-%m-%Y %H:%M:%S")
print(f"Inizio operazione: {format1}.")

response = client.messages.create(
    model=THINKING_MODEL,
    max_tokens=16000,

    thinking={
        "type": "enabled",
        "budget_tokens": 10000
    },
    #
    messages=[{
        "role": "user",
        "content": logica
    }]
)
end_time = time.time()
durata = calcola_durata(end_time, start_time)
print(f"Tempo impiegato: {durata:.2f} secondi")

print(response.to_json())

In [ ]:
#new
for block in response.content:
    if block.type == "thinking":
        print(f"\nRiassunto del pensiero: {block.thinking}")
    elif block.type == "text":
        print(f"\nRisposta testuale: {block.text}")

##CLAUDE 4 EXTENDED THINKING

In [ ]:
# chiamata a claude 4

start_time = time.time()
dt_object = datetime.datetime.fromtimestamp(start_time)
format1 = dt_object.strftime("%d-%m-%Y %H:%M:%S")
print(f"Inizio operazione: {format1}.")

response = client.messages.create(
    model=CLAUDE4,
    max_tokens=16000,
    #proprietà thinking è tipica di questo modello
    thinking={
        "type": "enabled",
        "budget_tokens": 10000
    },
    #
    messages=[{
        "role": "user",
        "content": analisi
    }]
)
end_time = time.time()
durata = calcola_durata(end_time, start_time)
print(f"Tempo impiegato: {durata:.2f} secondi")

print(response.to_json())

In [ ]:
for block in response.content:
    if block.type == "thinking":
        print(f"\nRiassunto del pensiero: {block.thinking}")
    elif block.type == "text":
        print(f"\nRisposta: {block.text}")

In [ ]:
#esercizio
#reset dataframe
df_confronto_tempi = pd.DataFrame()

models=[THINKING_MODEL, CLAUDE4]
prompts=[
    ('numeri_primi', numeri_primi),
    ('logica', logica),

    ('analisi', analisi)
]
for m in models:
  start_time = time.time()
  dt_object = datetime.datetime.fromtimestamp(start_time)
  format1 = dt_object.strftime("%d-%m-%Y %H:%M:%S")
  print(f"Inizio operazione: {format1}.")
  for prompt_name, p in prompts:
    response = client.messages.create(
    model=m,
    max_tokens=16000,
    thinking={
        "type": "enabled",
        "budget_tokens": 10000
    },
    #
    messages=[{
        "role": "user",
        "content": p
    }]
    )
    end_time = time.time()
    durata = calcola_durata(end_time, start_time)
    print(f"Tempo impiegato: {durata:.2f} secondi")

    data_for_df.append({
          'Modello': m,
          'Prompt': prompt_name,
          'Tipo Thinking': 'Enabled',
          'Budget Thinking': 10000,
          'Durata (s)': durata
      })

    for block in response.content:
        if block.type == "thinking":
            print(f"\nRiassunto del pensiero: {block.thinking}")
        elif block.type == "text":
            print(f"\nRisposta testuale: {block.text}")

    time.sleep(1)


In [ ]:
df_confronto_tempi = pd.DataFrame(data_for_df)
df_confronto_tempi
#print("\nDataFrame Confronto Tempi Modelli:")
#print(df_confronto_tempi)

In [ ]:
response = client.messages.create(
    model=CLAUDE4,
    max_tokens=16000,

    thinking={
        "type": "enabled",
        "budget_tokens": 10000
    },
    #
    messages=[{
        "role": "user",
        "content":dimostrazione_potenze_due
    }]
)
end_time = time.time()
durata = calcola_durata(end_time, start_time)
print(f"Tempo impiegato: {durata:.2f} secondi")

print(response.to_json())

In [ ]:
for block in response.content:
      if block.type == "thinking":
          print(f"\nRiassunto del pensiero: {block.thinking}")
      elif block.type == "text":
          print(f"\nRisposta testuale: {block.text}")


**Fine**
